In [6]:
import numpy as np
from sklearn import preprocessing
import tensorflow as tf

# =============================================================================
# Zad 1
# =============================================================================

raw_csv_data = np.loadtxt('Audiobooks_data.csv', delimiter=',')

unscaled_inputs_all = raw_csv_data[:, 1:-1]
targets_all          = raw_csv_data[:, -1]

shuffled_indices = np.arange(unscaled_inputs_all.shape[0])
np.random.shuffle(shuffled_indices)

unscaled_inputs_all = unscaled_inputs_all[shuffled_indices]
targets_all         = targets_all[shuffled_indices]

num_one_targets      = int(np.sum(targets_all))
zero_targets_counter = 0
indices_to_remove    = []

for i in range(targets_all.shape[0]):
    if targets_all[i] == 0:
        zero_targets_counter += 1
        if zero_targets_counter > num_one_targets:
            indices_to_remove.append(i)

unscaled_inputs_equal_priors = np.delete(unscaled_inputs_all, indices_to_remove, axis=0)
targets_equal_priors         = np.delete(targets_all,         indices_to_remove, axis=0)

scaled_inputs = preprocessing.scale(unscaled_inputs_equal_priors)

shuffled_indices = np.arange(scaled_inputs.shape[0])
np.random.shuffle(shuffled_indices)

shuffled_inputs  = scaled_inputs[shuffled_indices]
shuffled_targets = targets_equal_priors[shuffled_indices]

samples_count           = shuffled_inputs.shape[0]
train_samples_count      = int(0.8 * samples_count)
validation_samples_count = int(0.1 * samples_count)
test_samples_count       = samples_count - train_samples_count - validation_samples_count

train_inputs      = shuffled_inputs[:train_samples_count]
train_targets     = shuffled_targets[:train_samples_count]

validation_inputs  = shuffled_inputs[train_samples_count:train_samples_count + validation_samples_count]
validation_targets = shuffled_targets[train_samples_count:train_samples_count + validation_samples_count]

test_inputs  = shuffled_inputs[train_samples_count + validation_samples_count:]
test_targets = shuffled_targets[train_samples_count + validation_samples_count:]

print("Zbiór treningowy   :", np.sum(train_targets),      train_samples_count,      np.sum(train_targets)      / train_samples_count)
print("Zbiór walidacyjny  :", np.sum(validation_targets), validation_samples_count, np.sum(validation_targets) / validation_samples_count)
print("Zbiór testowy      :", np.sum(test_targets),       test_samples_count,       np.sum(test_targets)       / test_samples_count)

np.savez('Audiobooks_data_train',      inputs=train_inputs,      targets=train_targets)
np.savez('Audiobooks_data_validation', inputs=validation_inputs, targets=validation_targets)
np.savez('Audiobooks_data_test',       inputs=test_inputs,       targets=test_targets)

print("\nPliki .npz zapisane.\n")

# =============================================================================
# Zad 2
# =============================================================================

npz = np.load('Audiobooks_data_train.npz')
train_inputs  = npz['inputs'].astype(np.float32)
train_targets = npz['targets'].astype(np.int32)

npz = np.load('Audiobooks_data_validation.npz')
validation_inputs, validation_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

pz = np.load('Audiobooks_data_test.npz')
test_inputs, test_targets = npz['inputs'].astype(np.float32), npz['targets'].astype(np.int32)

input_size        = 10
output_size       = 2
# hidden_layer_size = 50
hidden_layer_size = 200
model = tf.keras.Sequential([
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),   # warstwa ukryta 1
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),   # warstwa ukryta 2
    tf.keras.layers.Dense(output_size,       activation='softmax') # warstwa wyjściowa
])

"""
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)"""

model.compile(
    optimizer='adamw',
    loss='sparse_categorical_crossentropy',
    metrics = ['accuracy']
)


batch_size    = 100
max_epochs    = 100
early_stopping = tf.keras.callbacks.EarlyStopping(patience=2)

model.fit(
    train_inputs,
    train_targets,
    batch_size=batch_size,
    epochs=max_epochs,
    callbacks=[early_stopping],
    validation_data=(validation_inputs, validation_targets),
    verbose=1
)

test_loss, test_accuracy = model.evaluate(test_inputs, test_targets)
print(f'\nTest loss: {test_loss:.2f}. Test accuracy: {test_accuracy * 100:.2f}%')

Zbiór treningowy   : 1791.0 3579 0.5004191114836547
Zbiór walidacyjny  : 228.0 447 0.5100671140939598
Zbiór testowy      : 218.0 448 0.48660714285714285

Pliki .npz zapisane.

Epoch 1/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.7239 - loss: 0.5181 - val_accuracy: 0.7360 - val_loss: 0.4472
Epoch 2/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7754 - loss: 0.4248 - val_accuracy: 0.7830 - val_loss: 0.4064
Epoch 3/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7832 - loss: 0.4035 - val_accuracy: 0.7875 - val_loss: 0.3934
Epoch 4/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7885 - loss: 0.3924 - val_accuracy: 0.7718 - val_loss: 0.3880
Epoch 5/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7899 - loss: 0.3848 - val_accuracy: 0.8210 - val_loss: 0.3731
Epoch 6/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7932 - loss: 0.3814 - val_accuracy: 0.7964 - val_loss: 0.3785
Epoch 7/100
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy